# Скалярное автоматическое дифференцирование и обучение одного нейрона

Лабораторная работа № 3. Ячейки выполняются сверху вниз. Для кода достаточно стандартной библиотеки Python.

Учебные материалы Дмитрия Власова. [Методичка](./lab_guide.pdf) · [Теория](./theory.pdf)

## Подготовка

`assert_close` проверяет абсолютную погрешность вычислений с числами с плавающей точкой.

In [1]:
# AUTOGRAD_ESSENTIALS_CELL: imports_and_helper
import math


def assert_close(actual, expected, tolerance=1e-9):
    difference = abs(actual - expected)
    assert difference <= tolerance, (
        f"Ожидалось {expected}, получено {actual}, "
        f"абсолютная ошибка {difference}"
    )

## Часть первая. Класс `Value`

Каждый объект хранит число `data`, градиент `grad`, ссылки на входы операции `parents`, метку `op` и локальное правило `_backward`. Проверка в конце ячейки создаёт лист графа со значением 2 и начальным градиентом 0.

In [2]:
# AUTOGRAD_ESSENTIALS_CELL: value
class Value:

    def __init__(self, data, parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self.parents = parents
        self.op = op
        self._backward = lambda: None

    def __repr__(self):
        return (
            f"Value(data={self.data:.6g}, grad={self.grad:.6g}, "
            f"op={self.op!r})"
        )

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        result = Value(
            self.data + other.data,
            parents=(self, other),
            op="+",
        )
        def backward():
            self.grad += result.grad
            other.grad += result.grad
        result._backward = backward
        return result
    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        result = Value(
            self.data * other.data,
            parents=(self, other),
            op="*",
        )
        def backward():
            self.grad += result.grad * other.data
            other.grad += result.grad * self.data
        result._backward = backward
        return result
    __rmul__ = __mul__

    def __neg__(self):
        result = Value(-self.data, parents=(self,), op="neg")
        def backward():
            self.grad += result.grad * (-1)
        result._backward = backward
        return result

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, power):
        result = Value(
            self.data ** power,
            parents=(self,),
            op=f"pow({power})",
        )
        def backward():
            local_derivative = power * self.data ** (power - 1)
            self.grad += result.grad * local_derivative
        result._backward = backward
        return result

    def sin(self):
        result = Value(math.sin(self.data), parents=(self,), op="sin")
        def backward():
            self.grad += result.grad * math.cos(self.data)
        result._backward = backward
        return result

    def exp(self):
        result = Value(math.exp(self.data), parents=(self,), op="exp")
        def backward():
            self.grad += result.grad * result.data
        result._backward = backward
        return result

    def log(self):
        result = Value(math.log(self.data), parents=(self,), op="log")
        def backward():
            self.grad += result.grad * (1 / self.data)
        result._backward = backward
        return result

    def tanh(self):
        result = Value(math.tanh(self.data), parents=(self,), op="tanh")
        def backward():
            self.grad += result.grad * (1 - result.data ** 2)
        result._backward = backward
        return result

    def backward(self):
        topo = []
        visited = set()
        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build(parent)
                topo.append(node)
        build(self)
        for node in topo:
            node.grad = 0.0
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


first_value = Value(2)
assert first_value.data == 2.0
assert first_value.grad == 0.0
assert first_value.parents == ()
assert first_value.op == ""
print(first_value)

Value(data=2, grad=0, op='')


## Прямой проход: сложение


In [3]:
# AUTOGRAD_ESSENTIALS_CELL: addition_experiment
left = Value(2.5)
right = Value(-1.0)
sum_value = left + right
assert_close(sum_value.data, 1.5)
assert sum_value.parents == (left, right)
assert sum_value.op == "+"
print("Сложение в прямом проходе: OK")

Сложение в прямом проходе: OK


## Прямой проход: умножение


In [4]:
# AUTOGRAD_ESSENTIALS_CELL: multiplication_experiment
left = Value(4.0)
right = Value(-0.5)
product_value = left * right
scaled_value = 3.0 * left
assert_close(product_value.data, -2.0)
assert_close(scaled_value.data, 12.0)
assert product_value.parents == (left, right)
assert product_value.op == "*"
print("Умножение в прямом проходе: OK")

Умножение в прямом проходе: OK


## Прямой проход: отрицание, вычитание и степень


In [5]:
# AUTOGRAD_ESSENTIALS_CELL: neg_sub_power_experiment
source = Value(3.0)
negative = -source
difference = 5.0 - source
cube = source ** 3
assert_close(negative.data, -3.0)
assert_close(difference.data, 2.0)
assert_close(cube.data, 27.0)
assert negative.op == "neg"
assert cube.op == "pow(3)"
print("Отрицание, вычитание и степень в прямом проходе: OK")

Отрицание, вычитание и степень в прямом проходе: OK


## Прямой проход: элементарные функции


In [6]:
# AUTOGRAD_ESSENTIALS_CELL: elementary_experiment
angle = Value(0.0)
positive = Value(2.0)
activation_input = Value(0.7)
assert_close(angle.sin().data, 0.0)
assert_close(positive.exp().log().data, 2.0)
assert_close(activation_input.tanh().data, math.tanh(0.7))
print("Элементарные функции в прямом проходе: OK")

Элементарные функции в прямом проходе: OK


## Обратный проход и общий узел графа


In [7]:
# AUTOGRAD_ESSENTIALS_CELL: backward_shared_graph
a = Value(2.0)
objective = a * a + a
objective.backward()
assert_close(objective.data, 6.0)
assert_close(a.grad, 5.0)
assert_close(objective.grad, 1.0)
print("Обратный проход и накопление двух вкладов: OK")

print("Значение выражения при a = 2:", objective.data)
print("Производная по a:", a.grad)

objective.backward()
assert_close(a.grad, 5.0)
print("Повторный backward() сохраняет градиент 5: OK")

Обратный проход и накопление двух вкладов: OK
Значение выражения при a = 2: 6.0
Производная по a: 5.0
Повторный backward() сохраняет градиент 5: OK


## Ручная трассировка локального обратного прохода

Для $a=0.5$, $b=1$ строим $m=ab$, $s=m+a$, $J=\tanh(s)$. На схеме стрелки показывают, как значения передаются при прямом проходе.

```text
a ──┬──→ m ──→ s ──→ J
    │    ↑     ↑
    │    b     │
    └──────────┘
```

Обратный проход начинается с $g_J=1$. После шага $J$ имеем $g_s=1-J^2$. Шаг $s$ передаёт $g_s$ в $m$ и $a$. Шаг $m$ прибавляет $b g_s$ к $g_a$ и $a g_s$ к $g_b$. Последняя проверка сравнивает ручную трассировку с автоматическим обходом.

In [8]:
# AUTOGRAD_ESSENTIALS_CELL: manual_backward_trace
a = Value(0.5)
b = Value(1.0)
m = a * b
s = m + a
j = s.tanh()
for node in (a, b, m, s, j):
    node.grad = 0.0
j.grad = 1.0
expected_gs = 1.0 - j.data ** 2
j._backward()
assert_close(s.grad, expected_gs)
assert_close(m.grad, 0.0)
assert_close(a.grad, 0.0)
print("После J._backward():", "g_s =", s.grad)
s._backward()
assert_close(m.grad, expected_gs)
assert_close(a.grad, expected_gs)
print("После s._backward():", "g_m =", m.grad, "; первый вклад в g_a =", a.grad)
m._backward()
assert_close(a.grad, expected_gs * (1.0 + b.data))
assert_close(b.grad, expected_gs * a.data)
print("После m._backward():", "g_a =", a.grad, "; g_b =", b.grad)

j.backward()
assert_close(s.grad, expected_gs)
assert_close(m.grad, expected_gs)
assert_close(a.grad, expected_gs * (1.0 + b.data))
assert_close(b.grad, expected_gs * a.data)
print("Ручная трассировка совпадает с backward(): OK")

После J._backward(): g_s = 0.41997434161402614
После s._backward(): g_m = 0.41997434161402614 ; первый вклад в g_a = 0.41997434161402614
После m._backward(): g_a = 0.8399486832280523 ; g_b = 0.20998717080701307
Ручная трассировка совпадает с backward(): OK


## Численная проверка центральной разностью


In [9]:
# AUTOGRAD_ESSENTIALS_CELL: finite_difference


def plain_function(x):
    return math.sin(x) * math.exp(x) + x ** 3


point = 0.37
step = 1e-6
input_value = Value(point)
objective = input_value.sin() * input_value.exp() + input_value ** 3
objective.backward()
numerical_gradient = (
    plain_function(point + step) - plain_function(point - step)
) / (2 * step)
absolute_error = abs(input_value.grad - numerical_gradient)
assert absolute_error < 1e-5, absolute_error
print("Градиент Value:", input_value.grad)
print("Градиент разностью:", numerical_gradient)
print("Абсолютная ошибка:", absolute_error)

Градиент Value: 2.2839857484831385
Градиент разностью: 2.283985748363637
Абсолютная ошибка: 1.195012977461829e-10


## Часть вторая: один нейрон с гиперболическим тангенсом


In [10]:
# AUTOGRAD_ESSENTIALS_CELL: tanh_neuron
class TanhNeuron:
    def __init__(self, weight, bias):
        self.weight = Value(weight)
        self.bias = Value(bias)

    def __call__(self, x):
        return (self.weight * x + self.bias).tanh()

    def parameters(self):
        return [self.weight, self.bias]

## Прямой проход нейрона


In [11]:
# AUTOGRAD_ESSENTIALS_CELL: neuron_forward_experiment
neuron = TanhNeuron(weight=2.0, bias=-1.0)
prediction = neuron(0.5)
assert_close(prediction.data, 0.0)
assert neuron.parameters() == [neuron.weight, neuron.bias]
print("Прямой проход одного нейрона: OK")

Прямой проход одного нейрона: OK


## Среднеквадратичная ошибка и проверка градиентов параметров

Нейрон вычисляет $\hat y_\mu=\tanh(wx_\mu+b)$, а ошибка равна $L=\frac{1}{n}\sum_{\mu=1}^n(\hat y_\mu-y_\mu)^2$. Индекс $\mu$ перебирает объекты обучающей выборки. В Python пары входов и целей даёт `zip(inputs, targets)`.

In [12]:
# AUTOGRAD_ESSENTIALS_CELL: loss_and_gradient
INPUTS = [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0]
TARGETS = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0]


def mean_squared_loss(neuron, inputs, targets):
    total = Value(0.0)
    # Немая сумма по объектам mu; zip выдаёт очередные значения без индекса mu.
    for input_number, target_number in zip(inputs, targets):
        prediction = neuron(input_number)
        error = prediction - target_number
        total = total + error ** 2
    return total * (1.0 / len(inputs))


def plain_loss(weight, bias, inputs, targets):
    total = 0.0
    # Та же немая сумма по объектам mu, но без построения графа Value.
    for input_number, target_number in zip(inputs, targets):
        prediction = math.tanh(weight * input_number + bias)
        total += (prediction - target_number) ** 2
    return total / len(inputs)


gradient_neuron = TanhNeuron(weight=0.0, bias=0.0)
initial_loss = mean_squared_loss(gradient_neuron, INPUTS, TARGETS)
initial_loss.backward()
step = 1e-6
numerical_weight_gradient = (
    plain_loss(step, 0.0, INPUTS, TARGETS)
    - plain_loss(-step, 0.0, INPUTS, TARGETS)
) / (2 * step)
numerical_bias_gradient = (
    plain_loss(0.0, step, INPUTS, TARGETS)
    - plain_loss(0.0, -step, INPUTS, TARGETS)
) / (2 * step)
assert_close(initial_loss.data, 1.0)
assert_close(
    gradient_neuron.weight.grad,
    numerical_weight_gradient,
    tolerance=1e-5,
)
assert_close(
    gradient_neuron.bias.grad,
    numerical_bias_gradient,
    tolerance=1e-5,
)
print("Начальная ошибка:", initial_loss.data)
print("Градиент по весу:", gradient_neuron.weight.grad)
print("Градиент по смещению:", gradient_neuron.bias.grad)

print("Численный градиент по весу:", numerical_weight_gradient)
print("Численный градиент по смещению:", numerical_bias_gradient)

Начальная ошибка: 1.0
Градиент по весу: -3.0
Градиент по смещению: 0.6666666666666666
Численный градиент по весу: -2.9999999999197335
Численный градиент по смещению: 0.6666666666488297


## Градиентный спуск и диагностика обучения

На каждой эпохе строим новый граф MSE на шести объектах, вызываем `loss.backward()` и обновляем вес и смещение. После 80 обновлений повторно измеряем MSE и определяем классы по знаку выхода нейрона.

In [13]:
# AUTOGRAD_ESSENTIALS_CELL: train_neuron
learning_rate = 0.2
epochs = 80
trained_neuron = TanhNeuron(weight=0.0, bias=0.0)
losses = []
for epoch in range(epochs):
    loss = mean_squared_loss(trained_neuron, INPUTS, TARGETS)
    loss.backward()
    losses.append(loss.data)
    for parameter in trained_neuron.parameters():
        parameter.data -= learning_rate * parameter.grad

final_predictions = [
    trained_neuron(input_number).data
    for input_number in INPUTS
]
predicted_labels = [
    1.0 if prediction >= 0.0 else -1.0
    for prediction in final_predictions
]
accuracy = sum(
    prediction == target
    for prediction, target in zip(predicted_labels, TARGETS)
) / len(TARGETS)
final_loss = mean_squared_loss(trained_neuron, INPUTS, TARGETS).data

assert final_loss < 0.03, final_loss
assert losses[-1] < 0.03, losses[-1]
assert accuracy == 1.0, accuracy
assert abs(trained_neuron.weight.data) > 0.1
assert abs(trained_neuron.bias.data) > 0.1
print("Ошибка в первой эпохе:", losses[0])
print("Ошибка перед обновлением в последней эпохе:", losses[-1])
print("Вес после обучения:", trained_neuron.weight.data)
print("Смещение после обучения:", trained_neuron.bias.data)
print("Прогнозы:", final_predictions)
print("Точность:", accuracy)

print("Ошибка после 80 обновлений:", final_loss)
print("Метки классов:", [int(label) for label in predicted_labels])

Ошибка в первой эпохе: 1.0
Ошибка перед обновлением в последней эпохе: 0.025550030181410055
Вес после обучения: 1.8495130666687565
Смещение после обучения: -0.8591099191612305
Прогнозы: [-0.9999945622967157, -0.9997802971885478, -0.9911605296754576, -0.6957987963048693, 0.7575341747647283, 0.9931949722390057]
Точность: 1.0
Ошибка после 80 обновлений: 0.02524209027898801
Метки классов: [-1, -1, -1, -1, 1, 1]


## Чек-лист

- [x] Проверены конструктор и прямой проход всех операций из методички.
- [x] Градиент для `a * a + a` при `a = 2` равен 5 при каждом вызове `backward()`.
- [x] Ручная трассировка совпадает с автоматическим обратным проходом.
- [x] Центральная разность подтверждает градиент скалярной функции, веса и смещения.
- [x] MSE после обучения меньше 0.03, точность на шести обучающих объектах равна 1.

## Краткий конспект

- На прямом проходе операции создают узлы ориентированного ациклического графа. На обратном проходе `backward()` обнуляет градиенты, задаёт $g_J=1$ и обходит узлы от результата к входам в обратном топологическом порядке.
- Для каждого входа $u$ операции $z$ правило цепочки даёт вклад $g_u\leftarrow g_u+g_z\,\partial z/\partial u$. Ветви используют один и тот же объект `Value`, поэтому вклады складываются в его `grad`.
- Центральная разность $[f(x+h)-f(x-h)]/(2h)$ при $h=10^{-6}$ независимо проверяет производную. Для $f(x)=\sin(x)\exp(x)+x^3$ при $x=0.37$ градиент равен **2.283985748**, абсолютное расхождение **1.20e-10**. При $w=b=0$ градиенты MSE равны $-3$ по весу и $2/3$ по смещению.
- После 80 обновлений с шагом $\eta=0.2$ MSE снизилась с **1.000000** до **0.025242**. Получены $w\approx1.849513$ и $b\approx-0.859110$. На обучающем наборе правильно классифицированы **6 из 6** объектов.

## Ответы на вопросы для самопроверки

1. Вклад приходит через произведение $a^2$ и через прямой вход $a$ в сложение. Произведение даёт $2a$, а сложение даёт $1$. При $a=2$ сумма равна $2+2+1=5$.
2. Начальный градиент равен единице, поскольку $\partial J/\partial J=1$.
3. В частной производной $\partial(xy)/\partial x=y$ второй множитель фиксирован. Поэтому локальный градиент по $x$ масштабируется значением $y$, и аналогично для $y$.
4. Обновление веса перед обратным проходом меняет значение, которое локальные правила могут прочитать из графа. Сначала вычисляем производные при параметрах прямого прохода, затем обновляем параметры.
5. Выход $\tanh$ лежит в $(-1,1)$ и приближается к $-1$ и $1$ при больших по модулю входах соответствующего знака. Класс можно восстановить по знаку выхода.
6. Учебная реализация поддерживает скаляры и производные первого порядка. За её рамками находятся тензоры, векторизованные батчи, broadcasting, производные высших порядков и готовые оптимизаторы.
7. `visited` добавляет в порядок обхода один объект $a$. Кортеж родителей произведения содержит две ссылки `(a, a)`, и локальная функция умножения прибавляет градиент по каждому из двух аргументов.
8. Умножение сохраняет ссылки на `self`, `other` и `result`. При вызове замыкание читает текущий `result.grad` и значения множителей. Поэтому `parameter.data` меняется после `loss.backward()`, а для следующей эпохи строится новый граф.